In [ ]:
from torch.nn import *
import torch

torch.set_default_tensor_type(torch.cuda.FloatTensor)


def numel(m):
    return sum(m.numel() for m in m.parameters())


def hide_from_numel(i, o, n):
    return round((n - o) / (i + o + 1))

In [ ]:
b = 65536 * 16
m1 = 1
m1 = Sequential(Linear(4, 4 * m1), Tanh(), Linear(4 * m1, 4 * m1), Tanh(), Linear(4 * m1, 4), BatchNorm1d(4, affine=0, track_running_stats=0))
m2 = hide_from_numel(4, 4, numel(m1) ** 2)
m2 = Sequential(Linear(m1[0].in_features, m2), Tanh(), Linear(m2, m1[-1].num_features), BatchNorm1d(4, affine=0, track_running_stats=0))
o = torch.optim.Adam((*m1.parameters(), *m2.parameters()))
s = torch.cuda.amp.GradScaler()

In [ ]:
while 1:
    n1 = n2 = L1 = L2 = 0
    for t in range(1000):
        l1 = -1
        while abs(l1 - t) > .01:
            n1 += 1
            l1 = t
            o.zero_grad()
            with torch.cuda.amp.autocast():
                t = torch.randn(b, m1[0].in_features)
                with torch.no_grad():
                    t2 = m1(t)
                t = functional.mse_loss(m2(t), t2)
            s.scale(t).backward()
            s.step(o)
            s.update()
        L1 += t.detach()
        l2 = 0
        t = 1
        while abs(l2 - t) > .01:
            n2 += 1
            l2 = t
            o.zero_grad()
            with torch.cuda.amp.autocast():
                t = torch.randn(b, m1[0].in_features)
                with torch.no_grad():
                    t2 = m2(t)
                t = functional.mse_loss(m1(t), t2)
            s.scale(-t).backward()
            s.step(o)
            s.update()
        L2 += t.detach()
    print(n1 / 1000, L1 / 1000, n2 / 1000, L2 / 1000)

In [ ]:
from sklearn.decomposition import PCA

t = torch.randn(65536 * 2, 4)
with torch.no_grad():
    t2 = PCA(2).fit_transform(m2(t).cpu())
    t = PCA(2).fit_transform(m1(t).cpu())

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(*t.T, label='1')
plt.scatter(*t2.T, label='2')
plt.legend()